In [ ]:
import numpy as np
import json
import pickle
from matplotlib import pyplot as plt
from scipy.special import softmax
from scipy.special import log_softmax

In [ ]:
logits = '/stor/work/Ellington/bioml/zero_shot_analysis/data/lysozyme/esmc_600m.logits.x.lysozyme.pkl'
with open(logits,'rb') as inf:
    logits = pickle.load(inf)
logits = logits['lysozyme.esmc_600m.logits.indices_1-']
wt_sequence = 'MKALIVLGLVLLSVTVQGKVFERCELARTLKRLGMDGYRGISLANWMCLAKWESGYNTRATNYNAGDRSTDYGIFQINSRYWCNDGKTPGAVNACHLSCSALLQDNIADAVACAKRVVRDPQGIRAWVAWRNRCQNRDVRQYVQGCGV'

In [ ]:
amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

def plot_position_distributions(logits, amino_acids, wt_sequence, positions, cols=4):
    """
    Plot softmax probability distributions for positions,
    highlighting the wild-type amino acid at each position.

    Args:
        logits:       numpy array of shape (N, 20)
        amino_acids:  list of 20 amino acid labels
        wt_sequence:  string of length N, e.g. "MKTAYIAKQRQISFVK..."
        pos_start:    first position to plot (0-indexed)
        pos_end:      last position to plot (exclusive)
        cols:         number of columns in the grid
    """
    positions = [x-1 for x in positions]
    n = len(positions)
    fig, axes = plt.subplots(n, 1, figsize=(6, n * 2.5))
    axes = np.array(axes).flatten()

    for i, pos in enumerate(positions):
        probs = softmax(logits[pos])
        ax = axes[i]

        wt_aa = wt_sequence[pos].upper()
        colors = ["tomato" if aa == wt_aa else "steelblue" for aa in amino_acids]
        ax.bar(amino_acids, probs, color=colors, width=0.7)

        ax.set_title(f"Position {pos + 1} (WT: {wt_aa})", fontsize=9, fontweight="bold")
        ax.set_ylim(0, max(probs) * 1.25)
        ax.set_ylabel("Probability", fontsize=7)
        ax.tick_params(axis="x", labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()
plot_position_distributions(logits, amino_acids, wt_sequence, positions = [5,12,18])

In [ ]:
def plot_mutation_odds(logits, amino_acids, wt_sequence, positions):
    positions = [x - 1 for x in positions]
    n = len(positions)

    fig, axes = plt.subplots(n, 1, figsize=(6, n * 2.5))
    axes = np.array(axes).flatten()

    for i, pos in enumerate(positions):
        log_probs = log_softmax(logits[pos])
        ax = axes[i]

        wt_aa = wt_sequence[pos].upper()
        wt_idx = amino_acids.index(wt_aa)
        wt_log_prob = log_probs[wt_idx]

        mut_aas  = [aa for j, aa in enumerate(amino_acids) if j != wt_idx]
        log_odds = [log_probs[j] - wt_log_prob for j in range(len(amino_acids)) if j != wt_idx]
        x_pos    = range(len(mut_aas))

        ax.bar(x_pos, log_odds, color="steelblue", width=0.7, alpha=0.4)
        ax.scatter(x_pos, log_odds, color="steelblue", s=20, zorder=3)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(mut_aas)
        ax.axhline(0.0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

        ax.set_title(f"Position {pos + 1} (WT: {wt_aa})", fontsize=9, fontweight="bold")
        ax.set_ylabel("Log odds vs WT", fontsize=7)
        ax.tick_params(axis="x", labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

# Usage
plot_mutation_odds(logits, amino_acids, wt_sequence, positions=[5, 12, 18])